# Spark Execution Model and Architecture

1. What is apache spark?
    - A distributed computing platform.
2. What do we do with apache spark?
    - We create programs and we execute them on an Apache spark cluster
        - How to create Spark programs?
        - How to execute Spark programs?

## How to execute Spark programs?

We have to methods to run spark program.
- Interactive Clients
    - Web based notebooks and command-line spark shell are both interactive clients. They offer you an easy methods to run your spark programs.
    - Both of them allow you to run your spark program line by line and get the output back on your console.
    - Interactive clients are best suited for exploration purpose.
    - But ultimately all your explorations will endup in a fullfleged spark application.
    - [ spark-shell, Notebook ]
- Submit Job
    - You may want to develope a stream processing application or a batch processing job and Spark supports both.
    - For examplt: 
        - [ Stream processing application : Example ] You want to read a news feed as a continuous stream then you want to apply a Ml algorithem to figure out the type of users that might be intrested in each news and redirect the story to those users.
        - [ Batch job : Example ] We collect the data for 24 hrs and start a scheduled job to compute the watch time minutes for the last 24 hrs. Finally the outcome goes to the table and also appears on the dashboard. 
    - In both the senarios either in a Stream processing job or a batch processing job you must package your application and submit it to the Spark cluster for execution.
    - **For a production usecase** Submit job method is used to execute the spark program.
    - Apache spark comes with a **spark-submit** utility which allows us to submit our spark jobs to the cluster.
    - spark-submit is the most commonly used when we have to send our packaged spark application for execution to the spark cluster.
    - Databricks cloud will allow you to submit the notebook itself and you do not need to package your application and use the spark submit tool.
    - Most of the cloud based spark vendors will allow you to use their RESTApi or a web based interface to submit your packaged spark application and they internally take care of running the job in their spark cluster.
    - spark-submit is a universally accepted method and works in almost all cases.
    - [ spark-submit, Databricks, Notebook, RestAPI ] 

### How your program runs?
Spark is a distributed processing engine. 
#### -> How a Spark distributed processing model runs?
![spark_processing_model](images/spark_processing_model.png)
- Spark applies a master slave architecutre to every spark application.
- So when we submit our application to the spark it is going to create a master process for your application.
- This master process is then going to create a bunch of slaves to distribute the work and compute your job.
- You can think of slaves to be a runtime containers with some dedicated cpu and memory
- In spark terminology the Master = Driver and Slaves = Executor.
- I am not talking about the cluster the cluster itself might have a Master node and a bunch of Slave nodes.
- But those things are the part of the cluster and are managed by the cluster manager.
- I am talking about the application and the containers.
- So the Spark engine is going to ask for a container from the underlying cluster manager to start the driver process.
- Once the driver process starts , the driver process is again going to ask for some more containers to start the executor processes. This happens for each application.
- Now from here on out the Driver and the Executor containers are responsible for running the application code that you submitted and doing the job that you wanted.
- So log story short every Spark application applies a master slave architecture and runs independently on the cluster and that is how the spark is a distributed computing platform.

## How spark runs your application on a local machine when we do not have a cluster or cluster manager?
### How does spark runs on a local machine?
- You can execute a spark application on a local machine without even having a real cluster
- You can configure your application to run on variety of clusters 
- The Spark engine is compatible with the following clusters : 
    - local[n]
    - YARN
    - Kubernetes
    - Mesos
    - Standalone
- When I execute a spark program from my jupyter notebook running locally on my laptop it uses ```local[n]``` cluster manager commonly known as local cluster manager.
    - When I create a spark session like this ```spark = SparkSession.builder.appName("DfApp").getOrCreate()``` and I din't specify any master, Spark automatically defaults to local mode meaning : 
        - This tells Spark to run everything on my laptop and use all available cpu cores.
        - So the Spark driver and the executor(s) all run as local JVM processes on my machine.
        - I am not connecting to an external Spark cluster — instead, Spark is emulating a cluster locally.
    - If suppose I do something like this ```spark = SparkSession.builder.appName("CPU_Stress_Test").master("local[1]").getOrCreate()```
        - Then in this case ```local[1]``` which tells Spark to use only one cpu core.
        - In this case only the Driver will be created an it will have to do all the execution by itself. Nothing happens in parallel.
    - This local cluster manager is designed for running testing and debugging your spark application locally.
    - This technique is nothing but a simulation of the distributed client server architechture using multiple threads so that we can test our application locally.
## How spark runs your application when we use interactive clients when we don't submit the application to the spark cluster?
Spark application can run in one of the following modes. 
- Client Mode
    - ![spark_client_mode_architecture](images/spark_client_mode_architecture.png)
    - The client mode is designed for interactive clients such as spark shell and the notebooks.
    - In this case the Spark Driver process runs locally at the client's machine.
    - The Driver running locally on the client's machine connects to the cluster manager and starts all the executors on the cluster.
    - This is a powerful feature submitting queries and getting the results back to the client
    - This is how the Spark shell and Notebooks are working.
    - If you logout from the client's machine then your Driver dies then in that case all the Executors that it created on the cluster also dies.
    - Client mode is suitable for interactive work but its not suitable for long running jobs.
- Cluster Mode
    - ![spark_cuslter_mode](images/spark_cuslter_mode.png)
    - The cluster mode is designed to submit your application to the cluster and let it run.
    - In this mode everything runs on the cluster.
    - Once you submit your application to run in cluster mode you can log off from the client machine and your Driver is not impacted because instead of running on a client's machine it runs on the cluster
    - This mode is used when we have to execute long running job on the cluster. 

## Execution model - When to use what?
- The cluster manager is at the top, hence first thing to do is to decide which cluster manager to use.
- In most of the cases either local[n] or YARN is used.
- When you are working on an IDE or a jupyter notebook locally on your machine you will use local[n] cluster manager.
- You will be using YARN when you are running your spark application on a real cluster.
- You have two types of cluster:
    - On-Premise
    - On-Cloud
- Next select the Execution mode:
    - Client mode:
        - When running the spark program locally the client mode should be selected. The clusters are simulated here via the underlying JVM engine using threads and treating threads as a single node.
        - Execution tool : 
            - IDE, Notebook
    - Cluster mode:
        - When running the program locally cluster mode don't make any sense because the clusters are not present locally.
        - Execution tool : 
            - Notebook, Shell
        - When submitting a spark application on a real cluster you should use cluster mode.
            - Execution tool :
                - Spark Submit

### Some important information : 
- If you ever decide to run spark-shell for any reason in my case I am using pyspark all you have to do is run pyspark command in my linux terminal. <br>
Use this link here http://localhost:4040/jobs/ to monitor and investigate things about your application.
- To execute a pyspark application or a script using spark-submit method
    - ```spark-submit --master "local[*]" pyspark_test.py```

## Why use Log4j instead of standard python logging?